# 01 — Data Processing and Integrity Validation
This notebook rebuilds the data pipeline end-to-end using only production modules under `src/` and YAML parameters under `configs/base/data.yaml`.

> Scope: load raw + processed datasets, verify split correctness, enforce reproducibility, validate integrity, and test missing/corrupted-value handling policies.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.utils.config_loader import resolve_config, load_yaml
from src.utils.seed import set_global_seed

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing src/ and configs/.')

ROOT = find_repo_root(Path.cwd())
DATA_YAML = ROOT / 'configs' / 'base' / 'data.yaml'
BASE_CONFIG = resolve_config(root=str(ROOT))
DATA_CONFIG = load_yaml(DATA_YAML).get('data', {})

SEED = int(DATA_CONFIG.get('random_seed', BASE_CONFIG.get('training', {}).get('random_seed', 42)))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'ROOT={ROOT}')
print(f'SEED={SEED}')
print(f"Configured pairs={DATA_CONFIG.get('pairs', [])}")

2026-03-13 03:34:35 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


ROOT=c:\Users\Nabeel\Desktop\frl-trading-framework
SEED=42
Configured pairs=['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD']


## Configuration provenance
The processing logic below is parameterized from `configs/base/data.yaml` (paths, train/test ratio, timestamp schema, missing-value policy, duplicate policy).
This guarantees behavior changes flow from configuration, not hard-coded notebook constants.

In [2]:
cfg_row = {
    'raw_data_dir': DATA_CONFIG.get('raw_data_dir'),
    'processed_data_dir': DATA_CONFIG.get('processed_data_dir'),
    'timestamp_column': DATA_CONFIG.get('timestamp_column'),
    'ohlcv_columns': ', '.join(DATA_CONFIG.get('ohlcv_columns', [])),
    'train_ratio': DATA_CONFIG.get('train_ratio'),
    'missing_value_policy': DATA_CONFIG.get('missing_value_policy'),
    'duplicate_policy': DATA_CONFIG.get('duplicate_policy'),
    'output_format': DATA_CONFIG.get('output_format'),
}
display(pd.DataFrame([cfg_row]))

,raw_data_dir,processed_data_dir,timestamp_column,ohlcv_columns,train_ratio,missing_value_policy,duplicate_policy,output_format
0,data/raw,data/processed,timestamp,"open, high, low, close, volume",0.8,ffill_then_drop_residual,keep_last,parquet


## Build processed train/test splits via workflow
We call `src.workflows.data_workflow.run_data_workflow` so preprocessing and feature handling stay centralized in source code.
No preprocessing logic is re-implemented in this notebook.

In [3]:
from src.workflows.data_workflow import run_data_workflow

PAIRS = list(DATA_CONFIG.get('pairs', []))
processed_results = run_data_workflow(
    config=BASE_CONFIG,
    pairs=PAIRS,
    root=str(ROOT),
)

summary_rows = []
for pair in PAIRS:
    train_df = processed_results[pair]['train']
    test_df = processed_results[pair]['test']
    summary_rows.append({
        'pair': pair,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'train_start': train_df['timestamp'].iloc[0],
        'train_end': train_df['timestamp'].iloc[-1],
        'test_start': test_df['timestamp'].iloc[0],
        'test_end': test_df['timestamp'].iloc[-1],
    })

display(pd.DataFrame(summary_rows))

,pair,train_rows,test_rows,train_start,train_end,test_start,test_end
0,EURUSD,79951,20000,2010-02-25 12:00:00+00:00,2022-12-20 17:00:00+00:00,2022-12-20 18:00:00+00:00,2026-03-06 21:00:00+00:00
1,GBPUSD,79951,20000,2010-02-24 14:00:00+00:00,2022-12-20 06:00:00+00:00,2022-12-20 07:00:00+00:00,2026-03-06 21:00:00+00:00
2,USDJPY,79951,20000,2010-02-24 03:00:00+00:00,2022-12-20 17:00:00+00:00,2022-12-20 18:00:00+00:00,2026-03-06 21:00:00+00:00
3,AUDUSD,79951,20000,2010-02-22 19:00:00+00:00,2022-12-20 16:00:00+00:00,2022-12-20 17:00:00+00:00,2026-03-06 21:00:00+00:00


## Load raw and processed datasets with `src/data`
This section validates schema, dtypes, missing percentages, and value ranges for both raw CSV loads and persisted processed splits.

In [4]:
from src.data.loader import load_raw_pair, load_processed_split

expected_ohlcv = list(DATA_CONFIG.get('ohlcv_columns', ['open', 'high', 'low', 'close', 'volume']))
timestamp_col = DATA_CONFIG.get('timestamp_column', 'timestamp')

integrity_rows = []
for pair in PAIRS:
    raw_df = load_raw_pair(pair, raw_dir=BASE_CONFIG['data']['raw_data_dir'], timestamp_column=timestamp_col)
    train_df = load_processed_split(pair, 'train', processed_dir=BASE_CONFIG['data']['processed_data_dir'])
    test_df = load_processed_split(pair, 'test', processed_dir=BASE_CONFIG['data']['processed_data_dir'])

    # Core schema checks
    assert timestamp_col in raw_df.columns, f'{pair}: missing timestamp column in raw data'
    for col in expected_ohlcv:
        assert col in raw_df.columns, f'{pair}: missing {col} in raw data'

    # Timestamp dtype checks
    assert pd.api.types.is_datetime64_any_dtype(raw_df[timestamp_col]), f'{pair}: timestamp not datetime in raw'
    assert pd.api.types.is_datetime64_any_dtype(train_df[timestamp_col]), f'{pair}: timestamp not datetime in train'
    assert pd.api.types.is_datetime64_any_dtype(test_df[timestamp_col]), f'{pair}: timestamp not datetime in test'

    # Missingness and range checks
    raw_missing_pct = 100.0 * raw_df[expected_ohlcv].isna().mean().mean()
    train_missing_pct = 100.0 * train_df[expected_ohlcv].isna().mean().mean()
    test_missing_pct = 100.0 * test_df[expected_ohlcv].isna().mean().mean()
    assert (train_df[['open', 'high', 'low', 'close']] > 0).all().all(), f'{pair}: non-positive processed prices in train'
    assert (test_df[['open', 'high', 'low', 'close']] > 0).all().all(), f'{pair}: non-positive processed prices in test'
    assert (train_df['volume'] >= 0).all(), f'{pair}: negative volume in train'
    assert (test_df['volume'] >= 0).all(), f'{pair}: negative volume in test'

    integrity_rows.append({
        'pair': pair,
        'raw_shape': raw_df.shape,
        'train_shape': train_df.shape,
        'test_shape': test_df.shape,
        'raw_missing_pct_ohlcv': round(raw_missing_pct, 4),
        'train_missing_pct_ohlcv': round(train_missing_pct, 4),
        'test_missing_pct_ohlcv': round(test_missing_pct, 4),
        'raw_close_min': float(raw_df['close'].min()),
        'raw_close_max': float(raw_df['close'].max()),
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

,pair,raw_shape,train_shape,test_shape,raw_missing_pct_ohlcv,train_missing_pct_ohlcv,test_missing_pct_ohlcv,raw_close_min,raw_close_max
0,EURUSD,"(10000, 6)","(7951, 26)","(2000, 26)",0.0,0.0,0.0,1.18828,1.49324
1,GBPUSD,"(10000, 6)","(7951, 26)","(2000, 26)",0.0,0.0,0.0,1.42425,1.67420
2,USDJPY,"(10000, 6)","(7951, 26)","(2000, 26)",0.0,0.0,0.0,76.18400,94.94500
3,AUDUSD,"(10000, 6)","(7951, 26)","(2000, 26)",0.0,0.0,0.0,0.80928,1.10633


## Train/test split correctness checks
We verify chronology and ratio consistency against `data.train_ratio` from configuration.
The split must be temporal with no overlap or leakage.

In [5]:
target_ratio = float(DATA_CONFIG.get('train_ratio', 0.8))
split_rows = []

for pair in PAIRS:
    train_df = processed_results[pair]['train']
    test_df = processed_results[pair]['test']
    total = len(train_df) + len(test_df)
    observed_ratio = len(train_df) / max(total, 1)

    # No overlap and strict chronology
    assert train_df['timestamp'].max() < test_df['timestamp'].min(), f'{pair}: train/test timestamp overlap detected'
    assert train_df['timestamp'].is_monotonic_increasing, f'{pair}: train timestamps not increasing'
    assert test_df['timestamp'].is_monotonic_increasing, f'{pair}: test timestamps not increasing'
    assert abs(observed_ratio - target_ratio) <= 0.02, (
        f'{pair}: observed train ratio {observed_ratio:.4f} differs from config {target_ratio:.4f}'
    )

    split_rows.append({
        'pair': pair,
        'target_train_ratio': target_ratio,
        'observed_train_ratio': round(observed_ratio, 4),
        'train_last_timestamp': train_df['timestamp'].max(),
        'test_first_timestamp': test_df['timestamp'].min(),
    })

display(pd.DataFrame(split_rows))

,pair,target_train_ratio,observed_train_ratio,train_last_timestamp,test_first_timestamp
0,EURUSD,0.8,0.799,2011-06-03 17:00:00+00:00,2011-06-03 18:00:00+00:00
1,GBPUSD,0.8,0.799,2011-06-02 19:00:00+00:00,2011-06-02 20:00:00+00:00
2,USDJPY,0.8,0.799,2011-06-02 17:00:00+00:00,2011-06-02 18:00:00+00:00
3,AUDUSD,0.8,0.799,2011-06-02 12:00:00+00:00,2011-06-02 13:00:00+00:00


## Missing/corrupted-value handling policy validation
We intentionally inject corruption and validate that source preprocessing behavior is consistent with configured policies (forward-fill, residual drop, duplicate handling, and value validation).

In [6]:
from src.data.preprocessing import preprocess_pair

example_pair = PAIRS[0]
example_raw = load_raw_pair(example_pair, raw_dir=BASE_CONFIG['data']['raw_data_dir'])
corrupted = example_raw.copy()

# Inject synthetic corruption: duplicate timestamp, missing OHLCV, and invalid price
corrupted.loc[0, 'close'] = np.nan
corrupted.loc[1, 'volume'] = np.nan
corrupted.loc[2, 'open'] = -1.0
corrupted = pd.concat([corrupted, corrupted.iloc[[3]]], ignore_index=True)

caught_error = None
try:
    _ = preprocess_pair(corrupted, BASE_CONFIG)
except ValueError as exc:
    caught_error = str(exc)

# Flag invalid rows, then rely on configured policy for imputation + residual-drop
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in corrupted.columns:
        corrupted.loc[corrupted[col] <= 0, col] = np.nan

train_clean, test_clean = preprocess_pair(corrupted, BASE_CONFIG)
assert train_clean[['open', 'high', 'low', 'close', 'volume']].isna().sum().sum() == 0
assert test_clean[['open', 'high', 'low', 'close', 'volume']].isna().sum().sum() == 0

print('Expected validation error before flagging:', caught_error)
print('Cleaned rows after policy handling:', len(train_clean) + len(test_clean))

Expected validation error before flagging: Non-positive values found in open
Cleaned rows after policy handling: 9999


In [7]:
final_summary = {
    'pairs_processed': len(PAIRS),
    'seed_used': SEED,
    'raw_dir': BASE_CONFIG['data']['raw_data_dir'],
    'processed_dir': BASE_CONFIG['data']['processed_data_dir'],
    'train_ratio': BASE_CONFIG['data']['train_ratio'],
    'missing_value_policy': BASE_CONFIG['data']['missing_value_policy'],
    'duplicate_policy': BASE_CONFIG['data']['duplicate_policy'],
}
display(pd.DataFrame([final_summary]))
print('✅ Data processing notebook checks passed end-to-end.')

,pairs_processed,seed_used,raw_dir,processed_dir,train_ratio,missing_value_policy,duplicate_policy
0,4,42,c:\Users\Nabeel\Desktop\frl-trading-framework\...,c:\Users\Nabeel\Desktop\frl-trading-framework\...,0.8,ffill_then_drop_residual,keep_last


✅ Data processing notebook checks passed end-to-end.
